# House Price Prediction — 2. Preprocessing & Feature Engineering

Builds a clean, model-ready dataset from the raw Zameen data based on the findings from EDA.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv(r'D:\Internship\house prediction\zameen-updated.csv',encoding = "latin1")
print('Raw shape:', df.shape)

Raw shape: (168446, 20)


## 1. Filter to 'For Sale' listings

In [3]:
df = df[df['purpose'] == 'For Sale'].copy()
print('After filtering to For Sale:', df.shape)

After filtering to For Sale: (120655, 20)


## 2. Drop irrelevant / unusable columns

In [4]:
drop_cols = ['property_id', 'location_id', 'page_url', 'purpose', 'agency', 'agent', 'Area Category']
df = df.drop(columns=drop_cols)
df.head()

,property_type,price,location,city,province_name,latitude,longitude,baths,area,bedrooms,date_added,Area Type,Area Size
0,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,4 Marla,2,02-04-2019,Marla,4.0
1,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,5.6 Marla,3,05-04-2019,Marla,5.6
2,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,8 Marla,5,07-17-2019,Marla,8.0
3,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,2 Kanal,4,04-05-2019,Kanal,2.0
4,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,3,07-10-2019,Marla,8.0


## 3. Unify area into a single unit (Marla)
1 Kanal = 20 Marla.

In [5]:
def to_marla(row):
    if row['Area Type'] == 'Kanal':
        return row['Area Size'] * 20
    return row['Area Size']

df['area_marla'] = df.apply(to_marla, axis=1)
df = df.drop(columns=['area', 'Area Type', 'Area Size'])
df[['area_marla']].describe()

,area_marla
count,120655.000000
mean,11.065215
std,59.560442
min,0.000000
25%,4.800000
50%,6.400000
75%,10.000000
max,12000.000000


## 4. Date feature engineering

In [6]:
df['date_added'] = pd.to_datetime(df['date_added'], format='%m-%d-%Y', errors='coerce')
df['listing_year'] = df['date_added'].dt.year
df['listing_month'] = df['date_added'].dt.month
df = df.drop(columns=['date_added'])
df[['listing_year', 'listing_month']].isnull().sum()

listing_year     0
listing_month    0
dtype: int64

In [7]:
df = df.dropna(subset=['listing_year', 'listing_month'])
df['listing_year'] = df['listing_year'].astype(int)
df['listing_month'] = df['listing_month'].astype(int)

## 5. Outlier removal

We remove extreme values that are almost certainly data entry errors (e.g. price of 0, absurdly small/large areas) using a percentile-based cutoff rather than arbitrary fixed thresholds, so it adapts to the data's own scale.

In [8]:
print('Before outlier removal:', df.shape)

# Remove non-positive prices
df = df[df['price'] > 0]

# Clip to 1st-99th percentile for price and area to remove extreme outliers
for col in ['price', 'area_marla']:
    low, high = df[col].quantile([0.01, 0.99])
    df = df[(df[col] >= low) & (df[col] <= high)]

# Remove unrealistic bedroom/bath counts (likely data errors)
df = df[(df['bedrooms'] <= 15) & (df['baths'] <= 15)]

print('After outlier removal:', df.shape)

Before outlier removal: (120655, 12)
After outlier removal: (116112, 12)


## 6. Log-transform the target

Based on EDA, price is heavily right-skewed. We model `log1p(price)` and will reverse this with `expm1()` after prediction.

In [9]:
df['log_price'] = np.log1p(df['price'])

## 7. Encode categorical features

- `property_type` (7 categories) and `city` (5 categories): one-hot encoding (low cardinality).
- `location` (hundreds of unique neighborhoods): **frequency encoding** — replacing each location with how often it appears in the training data. This avoids the dimensionality explosion of one-hot encoding while still letting the model use location as a numeric signal. We compute frequencies from the training set only, to avoid leaking test information.

In [10]:
print('Unique locations:', df['location'].nunique())
print('Unique cities:', df['city'].nunique())
print('Unique property types:', df['property_type'].nunique())
print('Unique provinces:', df['province_name'].nunique())

Unique locations: 1429
Unique cities: 5
Unique property types: 7
Unique provinces: 3


## 8. Train/test split
Split *before* frequency encoding to prevent data leakage from test set into training statistics.

In [11]:
feature_cols = ['property_type', 'city', 'province_name', 'location', 'latitude', 'longitude',
                'baths', 'bedrooms', 'area_marla', 'listing_year', 'listing_month']
target_col = 'log_price'

X = df[feature_cols].copy()
y = df[target_col].copy()
price_raw = df['price'].copy()

X_train, X_test, y_train, y_test, price_train, price_test = train_test_split(
    X, y, price_raw, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

Train: (92889, 11)  Test: (23223, 11)


In [12]:
# Frequency encoding for location (fit on train only)
location_freq = X_train['location'].value_counts(normalize=True)

X_train['location_freq'] = X_train['location'].map(location_freq)
X_test['location_freq'] = X_test['location'].map(location_freq).fillna(0)  # unseen locations -> 0

X_train = X_train.drop(columns=['location'])
X_test = X_test.drop(columns=['location'])

In [13]:
# One-hot encode property_type, city, province_name
cat_cols = ['property_type', 'city', 'province_name']
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Align test columns to train columns (handles any category mismatch)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.shape, X_test.shape)
X_train.head()

(92889, 20) (23223, 20)


,latitude,longitude,baths,bedrooms,area_marla,listing_year,listing_month,location_freq,property_type_Flat,property_type_House,property_type_Lower Portion,property_type_Penthouse,property_type_Room,property_type_Upper Portion,city_Islamabad,city_Karachi,city_Lahore,city_Rawalpindi,province_name_Punjab,province_name_Sindh
93864,25.071948,67.338656,3,3,8.0,2019,6,0.065153,False,True,False,False,False,False,False,True,False,False,False,True
48677,31.473035,73.195024,0,7,24.0,2019,3,0.000054,False,True,False,False,False,False,False,False,False,False,True,False
18123,31.390645,74.252887,3,3,5.0,2018,10,0.003316,False,True,False,False,False,False,False,False,True,False,True,False
59505,24.894076,67.027715,2,2,4.7,2019,4,0.065153,True,False,False,False,False,False,False,True,False,False,False,True
42237,31.465422,74.385309,4,4,10.0,2019,3,0.129402,False,True,False,False,False,False,False,False,True,False,True,False


## 9. Feature scaling
Scale numeric features for Linear Regression (tree-based models don't strictly need this, but it doesn't hurt them and keeps one consistent dataset).

In [14]:
from sklearn.preprocessing import StandardScaler

numeric_features = ['latitude', 'longitude', 'baths', 'bedrooms', 'area_marla',
                     'listing_year', 'listing_month', 'location_freq']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

## 10. Save processed data for modeling notebook

In [15]:
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
price_train.to_csv('../data/processed/price_train.csv', index=False)
price_test.to_csv('../data/processed/price_test.csv', index=False)

joblib.dump(scaler, '../data/processed/scaler.pkl')
joblib.dump(location_freq, '../data/processed/location_freq.pkl')

print('Saved all processed files to ../data/processed/')

Saved all processed files to ../data/processed/


## Summary

- Filtered to 'For Sale' listings only.
- Unified area into `area_marla`.
- Engineered `listing_year`/`listing_month` from date.
- Removed price=0 and 1st/99th percentile outliers in price & area.
- Target is `log1p(price)` to handle skew.
- `location` frequency-encoded (fit on train only, no leakage); `property_type`, `city`, `province_name` one-hot encoded.
- Numeric features standardized.
- Saved train/test splits (raw + scaled) to `../data/processed/`.

Next: **03_modeling.ipynb**